In [2]:
!pip install nltk scikit-learn pandas numpy matplotlib seaborn

In [3]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [4]:
import os

# Create folders
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/notebooks', exist_ok=True)
os.makedirs('/content/outputs', exist_ok=True)

In [6]:
import shutil

shutil.move('/content/IMDB Dataset.csv', '/content/data')

'/content/data/IMDB Dataset.csv'

In [8]:
import pandas as pd

df = pd.read_csv('/content/data/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [9]:
print("Shape:", df.shape)

print("\nClass Distribution:")
print(df['sentiment'].value_counts())

print("\nSample Data:")
df.sample(5)

Shape: (50000, 2)

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Data:


,review,sentiment
35244,I really loved it although while reading the r...,positive
40797,I had the good fortune of reading the book bef...,negative
39104,I saw this when on The Wonderful World of Disn...,positive
29151,When I saw this film the first time I was very...,positive
30982,"La Chute de la Maison Usher, or The Fall of th...",negative


In [10]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()  # lowercasing

    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-zA-Z]", " ", text)  # remove special characters

    tokens = text.split()  # tokenization

    tokens = [word for word in tokens if word not in stop_words]  # remove stopwords

    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # lemmatization

    return " ".join(tokens)

# Apply preprocessing
df['clean_text'] = df['review'].apply(preprocess_text)

df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


In [11]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text'])

X_bow.shape

(50000, 5000)

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text'])

X_tfidf.shape

(50000, 5000)

In [15]:
from sklearn.model_selection import train_test_split

y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

Model 1 Logistic Regression

In [16]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

LogisticRegression(max_iter=200)

Model 2 Naive Bayes

In [17]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

MultinomialNB()

Model 3 Decision Tree

In [18]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

DecisionTreeClassifier()

Evaluation Function

In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1 Score": f1_score(y_test, y_pred, average='weighted')
    }

evaluate all models

In [21]:
results = {
    "Logistic Regression": evaluate_model(lr, X_test, y_test),
    "Naive Bayes": evaluate_model(nb, X_test, y_test),
    "Decision Tree": evaluate_model(dt, X_test, y_test)
}

import pandas as pd

results_df = pd.DataFrame(results).T
results_df

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.8883,0.888544,0.8883,0.888269
Naive Bayes,0.8561,0.856142,0.8561,0.856087
Decision Tree,0.7167,0.716724,0.7167,0.716704


In [22]:
results_df.to_csv('/content/outputs/model_results.csv')

In [23]:
def predict_sentiment(text):
    cleaned = preprocess_text(text)
    vector = tfidf.transform([cleaned])
    prediction = lr.predict(vector)
    return prediction[0]

# Test
predict_sentiment("This movie was amazing and fantastic!")

'positive'

Insights:
TF-IDF gave better performance than BoW
Logistic Regression performed best overall
Naive Bayes is faster but slightly less accurate
Decision Tree overfitted the data